# Best Solution: Chronos-2 Embeddings + Handcrafted Features + XGBoost

**Pipeline:**
1. Extract mean-pooled embeddings from Chronos-2 (frozen, zero-shot) → 4608-dim
2. Compute handcrafted statistical features from raw time series → 90-dim
3. Concatenate both → 4698-dim feature vector
4. Train XGBoost with balanced sample weights

**Result: W-F1 = 0.680, Accuracy = 0.710** (vs MultiROCKET baseline W-F1 = 0.604)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix, accuracy_score,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tslearn.datasets import UCR_UEA_datasets
import seaborn as sns

SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"

# ── Load LSST dataset ──
X_train_raw, y_train_raw, X_test_raw, y_test_raw = (
    UCR_UEA_datasets().load_dataset("LSST")
)
X_train = np.transpose(X_train_raw, (0, 2, 1)).astype(np.float32)  # (n, 6, 36)
X_test  = np.transpose(X_test_raw,  (0, 2, 1)).astype(np.float32)

le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test  = le.transform(y_test_raw)
class_names = le.classes_
n_classes = len(class_names)
class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)

print(f"Train: {X_train.shape}, Test: {X_test.shape}, Classes: {n_classes}")
print(f"Class distribution (train): {np.bincount(y_train)}")

## Step 1: Chronos-2 Embedding Extraction

Chronos-2 is a time series foundation model with **native multivariate support**.
It processes all 6 photometric bands jointly via cross-channel attention,
capturing inter-band correlations that univariate models (MOMENT, TiReX) miss.

We use the frozen model as a feature extractor:
- Input: (6 channels, 36 timesteps) per sample
- Chronos-2 outputs: (6 channels, ~5 tokens, 768 hidden dim)
- Mean-pool over tokens → (6, 768) per sample
- Flatten → **4608-dimensional** embedding vector

In [ ]:
from chronos import Chronos2Pipeline

def extract_chronos_embeddings(X_cf, model_name="amazon/chronos-2",
                                batch_size=64, device="cuda"):
    """Extract mean-pooled embeddings from Chronos-2.
    
    Args:
        X_cf: (n, C, T) channels-first numpy array
    Returns:
        (n, C * d_model) = (n, 4608) numpy array
    """
    n = len(X_cf)
    X_cf = X_cf.astype(np.float32)
    pipeline = Chronos2Pipeline.from_pretrained(model_name, device_map=device)
    
    all_embeddings = []
    for i in range(0, n, batch_size):
        batch = [X_cf[j] for j in range(i, min(i + batch_size, n))]
        emb_list, _ = pipeline.embed(batch)
        for emb in emb_list:
            # emb: (C, num_tokens, 768) → mean over tokens → flatten
            pooled = emb.mean(dim=1).reshape(-1).cpu().numpy()
            all_embeddings.append(pooled)
        if (i // batch_size) % 5 == 0:
            print(f"  {min(i + batch_size, n)}/{n} samples")
    
    del pipeline; torch.cuda.empty_cache()
    return np.stack(all_embeddings)

emb_train = extract_chronos_embeddings(X_train, device=device)
emb_test  = extract_chronos_embeddings(X_test, device=device)
print(f"Embeddings: train={emb_train.shape}, test={emb_test.shape}")

## Step 2: Handcrafted Feature Engineering

Foundation model embeddings capture deep temporal patterns, but EDA revealed
that simple statistical properties are highly discriminative for LSST:
- **Amplitude** varies by 200x across classes (the most discriminative single feature)
- **Inter-band correlations** are class-specific (e.g., band g peaks before band r for certain transients)

We compute 90 features that explicitly encode these properties:
- **Per-channel** (6 × 10 = 60): mean, std, min, max, range, skew, kurtosis, slope, first value, last value
- **Cross-channel correlations** (15): pairwise Pearson correlations between all band pairs
- **Amplitude ratios** (15): log-scaled pairwise amplitude ratios between bands

In [ ]:
from scipy import stats as sp_stats

def compute_handcrafted_features(X_cf):
    """Extract 90 statistical features from raw (N, C, T) time series."""
    N, C, T = X_cf.shape
    feats = []
    
    for i in range(N):
        f = []
        # Per-channel statistics (6 channels × 10 features = 60)
        for c in range(C):
            ts = X_cf[i, c]
            f.extend([
                np.mean(ts), np.std(ts), np.min(ts), np.max(ts), np.ptp(ts),
                float(sp_stats.skew(ts)), float(sp_stats.kurtosis(ts)),
                np.polyfit(np.arange(T), ts, 1)[0],  # linear slope
                ts[0], ts[-1],
            ])
        
        # Pairwise cross-channel correlations (15 pairs)
        for c1 in range(C):
            for c2 in range(c1 + 1, C):
                r = np.corrcoef(X_cf[i, c1], X_cf[i, c2])[0, 1]
                f.append(r if np.isfinite(r) else 0.0)
        
        # Pairwise amplitude ratios (15 pairs)
        amps = [np.ptp(X_cf[i, c]) for c in range(C)]
        for c1 in range(C):
            for c2 in range(c1 + 1, C):
                f.append(np.log1p(amps[c1] / (amps[c2] + 1e-8)))
        
        feats.append(f)
    
    return np.nan_to_num(np.array(feats, dtype=np.float32))

hc_train = compute_handcrafted_features(X_train)
hc_test  = compute_handcrafted_features(X_test)
print(f"Handcrafted features: train={hc_train.shape}, test={hc_test.shape}")

## Step 3: Feature Concatenation + XGBoost Classification

Concatenate embeddings (4608) + handcrafted features (90) → 4698-dim input.

XGBoost is trained with **balanced sample weights** to handle the 111:1 class
imbalance (class 90 has 777 samples, class 53 has 7).

In [ ]:
# Concatenate features
X_tr = np.concatenate([emb_train, hc_train], axis=1)
X_te = np.concatenate([emb_test, hc_test], axis=1)
print(f"Final features: train={X_tr.shape}, test={X_te.shape}")

# Balanced sample weights
sample_weights = np.array([class_weights[y] for y in y_train])

# Train XGBoost
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    tree_method="hist",
    device="cuda",
    eval_metric="mlogloss",
)
model.fit(X_tr, y_train, sample_weight=sample_weights)
print("Training complete.")

## Evaluation

In [ ]:
def evaluate(y_true, y_pred, class_names=None, model_name="Model"):
    acc = accuracy_score(y_true, y_pred)
    f1_w = f1_score(y_true, y_pred, average="weighted")
    f1_m = f1_score(y_true, y_pred, average="macro")
    print(f"=== {model_name} ===")
    print(f"Accuracy:    {acc:.4f}")
    print(f"Weighted F1: {f1_w:.4f}")
    print(f"Macro F1:    {f1_m:.4f}")
    target_names = list(class_names) if class_names is not None else None
    print(classification_report(y_true, y_pred, target_names=target_names))
    return dict(accuracy=acc, weighted_f1=f1_w, macro_f1=f1_m)

y_pred = model.predict(X_te)
metrics = evaluate(y_test, y_pred, class_names, "Chronos-2 + HC + XGBoost")

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=class_names,
            yticklabels=class_names, cmap='Blues', ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Chronos-2 + Handcrafted Features + XGBoost')
plt.tight_layout()
plt.savefig('figures/best_solution_confmat.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Comparison with baselines
def results_table(results_dict):
    header = f"{'Model':<42} {'Acc':>8} {'W-F1':>8} {'M-F1':>8}"
    print(header)
    print("-" * len(header))
    for name, m in results_dict.items():
        print(f"{name:<42} {m['accuracy']:>8.4f} {m['weighted_f1']:>8.4f} {m['macro_f1']:>8.4f}")

comparison = {
    "MultiROCKET (best classical baseline)": dict(accuracy=0.6375, weighted_f1=0.6039, macro_f1=0.4615),
    "Chronos-2 + LogReg (embeddings only)": dict(accuracy=0.5998, weighted_f1=0.5984, macro_f1=0.4605),
    "Chronos-2 + HC + XGBoost (ours)": metrics,
}
results_table(comparison)

In [ ]:
# Feature importance: which feature groups matter most?
importances = model.feature_importances_
emb_importance = importances[:4608].sum()
hc_importance  = importances[4608:].sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Group-level importance
axes[0].bar(['Chronos-2 embeddings\n(4608 features)', 'Handcrafted\n(90 features)'],
            [emb_importance, hc_importance], color=['steelblue', 'coral'])
axes[0].set_ylabel('Total feature importance')
axes[0].set_title('Feature group importance')

# Top 20 handcrafted features
hc_names = []
bands = ['u', 'g', 'r', 'i', 'z', 'y']
for b in bands:
    for stat in ['mean', 'std', 'min', 'max', 'range', 'skew', 'kurt', 'slope', 'first', 'last']:
        hc_names.append(f'{b}_{stat}')
for i in range(6):
    for j in range(i+1, 6):
        hc_names.append(f'corr_{bands[i]}_{bands[j]}')
for i in range(6):
    for j in range(i+1, 6):
        hc_names.append(f'amp_{bands[i]}/{bands[j]}')

hc_imp = importances[4608:]
top_idx = np.argsort(hc_imp)[-15:][::-1]
axes[1].barh([hc_names[i] for i in top_idx], hc_imp[top_idx], color='coral')
axes[1].set_xlabel('Feature importance')
axes[1].set_title('Top 15 handcrafted features')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('figures/feature_importance.pdf', bbox_inches='tight')
plt.show()